In [1]:
try:
    %load_ext autoreload --quiet
except ImportError:
    %reload_ext autoreload
%autoreload 2

import hashlib
from typing import List, Any
from flow_merge.lib.snapshot.data_architecture._normalized_slices import NormalizedSlice
from pydantic import BaseModel, computed_field, Field
import datetime
from pathlib import Path
import json

class MergePlan(BaseModel):
    created_at: datetime.datetime = Field(default=datetime.datetime.now())
    base_model: str
    tokenizer_mode: str
    tokenizer_interpolation_method: str
    slices: List[NormalizedSlice]
    lib_version: str

    @classmethod
    def from_config(cls, config: Any) -> "MergePlan":
        return cls(
            created_at=datetime.datetime.now(),
            base_model=config.base_model,
            tokenizer_mode=config.tokenizer_mode,
            tokenizer_interpolation_method=config.tokenizer_interpolation_method,
            slices=config.slices,
            lib_version=config.lib_version,
        )

    @classmethod
    def from_file(cls, file_path: Path | str) -> "MergePlan":
        with open(Path(file_path).resolve(), "rb") as f:
            parsed = json.load(f)
            return cls(**parsed)

    @computed_field
    @property
    def sha(self) -> str:
        obj = self.model_dump_json(exclude={"created_at", "sha"}).encode("utf-8")
        return hashlib.md5(obj).hexdigest()

mp = MergePlan.from_file("../flow_merge/lib/example-slices.json")


In [2]:
print(mp.sha)

dacbc64dcbfa795dc7eb9a2c267bf051


In [9]:
from typing import Tuple
from flow_merge.lib.model import Model
from flow_merge.lib.tokenizer import get_merge_tokenizer

# get models from all the slices
all_model_path_occurrences: List[str|None] = [model_field.model for x in mp.slices for model_field in x.sources]
all_distinct_model_paths: List[str|None] = list({model for model in all_model_path_occurrences})
all_model_objects: List[Model] = [Model.from_path(model_path) for model_path in all_distinct_model_paths]

# what is the 'mode' for base model - which model is most frequently the base model
all_models: List[Tuple[str,bool]] = [(model_field.model, model_field.is_base) for x in mp.slices for model_field in x.sources]
the_most_frequent_base_model_name = max(set(name for name, is_base in all_models if is_base), 
                                   key=lambda name: sum(is_base for n, is_base in all_models if n == name), 
                                   default=None)
base_model = next((model for model in all_model_objects if model.id == the_most_frequent_base_model_name), None)

# construct merged tokenizer out of the distinct models
common_tokenizer = get_merge_tokenizer(models=all_model_objects, base_model=base_model, tokenizer_mode=mp.tokenizer_mode)


# id: ModelId
# path: Path
# metadata: ModelMetadata
# file_to_tensor_index: Optional[Dict]
# shards: List[ShardFile]
# architecture: ModelArchitecture

# get merge method from method config
# base_model: Model,
# task_base_model_weight: ModelWeight,
# task_models_with_weights: Dict[Model, ModelWeight],
# tokenizer: Tokenizer,
# method_config,
# sources



Loading Model from path: from_path
Creating metadata: _create_metadata
Loading Model Info: load_model_info
Fetching model info from HF: fetch_hf_model_info
Creating File Metadata_list from HF: create_file_metadata_list_from_hf
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Generating Content Hash: generate_content_hash
Missing model.safetensors.index.json file
Missing pytorch_model .bin files
Missing pytorch_model.bin.index.json file
Missing adapter files
Index files not found, using single shard file fallback.
Creating architecture: _create_architecture
Substitute the layer templates
Generating model weights: _generate_model_weights
Creating decoder weights: _cr

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


No differences in tokens or vocab among tokenizers. Using /home/ks/repos/new-july/flow-merge/notebooks/models/Qwen/Qwen1.5-0.5B for the tokenizer.


Tokenizer(tokenizer=Qwen2TokenizerFast(name_or_path='/home/ks/repos/new-july/flow-merge/notebooks/models/Qwen/Qwen1.5-0.5B', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}, input_ids_mappings=None)